# YOLOv1（PASCAL VOC 2007を用いた物体検出）

---
## 目的
物体検出を，画像全体に対する単一の回帰問題として定式化した最初期のSingle-Shot検出手法であるYOLOv1（You Only Look Once）を，PASCAL VOC 2007データセットを用いて実装する．S×Sのグリッドに画像を分割し，各セルが直接ボックス座標・confidence・クラス確率を予測する仕組みと，アンカーボックスを一切使わないその設計上のシンプルさについて理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import box_iou, nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して矩形領域（Bounding Box）が付与されたデータセットで，学習用（trainval）5011枚，評価用（test）4952枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCDetection`で読み込みます．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# YOLOv1はクラス確率をセルごとに「物体が存在する条件付き確率」として予測するため，
# SSDと異なり背景を明示的なクラスとして持たない（0〜19の20クラスのみ）
CLASS_TO_IDX = {name: i for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES)  # 背景クラスを持たない
IMG_SIZE = 448  # 原論文の入力解像度（448 / 7 = 64ピクセルごとに1グリッドセル）

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景なし）:', n_class)

## YOLOv1（You Only Look Once）
YOLOv1は，物体検出を「画像全体に対する1回の回帰問題」として定式化した，最初期のSingle-Shot型検出手法です．SSDのデフォルトボックスやFaster R-CNNの領域提案（Region Proposal）のような，あらかじめ大量に用意したアンカー（候補領域）を一切使いません（アンカーボックスが導入されたのはYOLOv2以降です）．かわりに，入力画像をS×Sのグリッド（本ノートブックではS=7）に分割し，各セルが直接「そのセル付近にある物体のボックス座標・confidence・クラス」を予測します．この仕組みのシンプルさが，本シリーズの他の検出手法（SSD・Faster R-CNNなど，より複雑なアンカー機構やRoI処理を持つ手法）との際立った対比になっています．

各グリッドセルは，次の情報を出力します．
- B個（本ノートブックではB=2）のボックス，それぞれ (x, y, w, h, confidence)
  - x, yはボックス中心の**セル内での相対位置**（0〜1，セルの左上を原点とする）であり，画像全体に対する相対位置ではない点に注意．
  - w, hは**画像全体に対する**相対的な幅・高さ（0〜1）．
  - confidenceは Pr(Object) × IoU(予測ボックス, 正解ボックス) で定義される，「物体らしさ×位置の正確さ」を表す単一のスカラー値．
- C個（=20，PASCAL VOCのクラス数）のクラス確率．**Bごとではなく，セルごとに1組だけ**予測される点がYOLOv1特有の重要な仕様です（後継のYOLOv2以降は，ボックス（アンカー）ごとにクラスを予測するように変更されています）．

したがって，1セルあたりの出力次元は B×5 + C = 2×5 + 20 = 30，ネットワーク全体の出力テンソルの形状は (S, S, B×5+C) = (7, 7, 30) となります．

### バックボーンと検出ヘッド
原論文は，24層の畳み込み層＋2層の全結合層からなる独自のDarknet系バックボーンを使用しています．本ノートブックでは，他の検出ノートブックと同様に事前学習済みResNet50を採用します．ResNet50の`layer4`までの出力はstride 32のため，448×448の入力に対して14×14の特徴マップになります．これをS=7に合わせるため，stride 2の畳み込みをもう1層追加して7×7までダウンサンプルし，最後に1×1畳み込みでB×5+C=30チャンネルの出力に変換します（原論文はここを全結合層で行いますが，全結合層を使わない全畳み込み構成でも同じ出力形状が得られるため，実装が簡潔な全畳み込みヘッドを採用します）．出力のうち(x, y, w, h, confidence)はSigmoidで，クラス確率はセルごとにSoftmaxをかけて0〜1に収めます（原論文は線形出力のままSSEを計算しますが，学習を安定させるため活性化関数を追加しています．差分は末尾の表にまとめます）．

In [ ]:
class YOLOv1(nn.Module):
    def __init__(self, S=7, B=2, C=20):
        super().__init__()
        self.S, self.B, self.C = S, B, C

        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # avgpool・fcを除いた畳み込み部分（448入力で14x14x2048）

        self.extra = nn.Sequential(
            # 14x14 -> 7x7 にダウンサンプルし，グリッドサイズSに合わせる
            nn.Conv2d(2048, 1024, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1, inplace=True),
        )

        self.head = nn.Sequential(
            nn.Dropout2d(0.5),  # 原論文同様，最終層の直前にDropoutを挿入
            nn.Conv2d(1024, B * 5 + C, kernel_size=1),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.extra(feat)
        out = self.head(feat)  # (N, B*5+C, S, S)
        out = out.permute(0, 2, 3, 1).contiguous()  # (N, S, S, B*5+C)

        box_out = out[..., :self.B * 5].view(-1, self.S, self.S, self.B, 5)
        box_out = torch.sigmoid(box_out)  # x, y, w, h, confidenceを全て[0, 1]に収める
        class_out = F.softmax(out[..., self.B * 5:], dim=-1)  # セルごとに共有する1組のクラス確率分布

        return torch.cat([box_out.view(-1, self.S, self.S, self.B * 5), class_out], dim=-1)  # (N, S, S, B*5+C)

### 正解ボックスのグリッドセルへの変換
学習時には，正解（Ground Truth）ボックスを，それぞれ「どのセルが責任を持つか」「そのセル内でのターゲット値」に変換する必要があります．各GTボックスについて，その中心座標が属するグリッドセルを求め（`row = int(cy / cell_size)`, `col = int(cx / cell_size)`），そのセルを「物体を含むセル」とします．ターゲット値は，予測値と同じ表現（x, yはセル内の相対位置，w, hは画像全体に対する相対サイズ）に変換して保持します．IoU計算のために，元のピクセル座標(xyxy)のボックスも合わせて保持しておきます．

なお，複数の物体の中心が同じセルに入ってしまった場合，YOLOv1はそのセルにつき1つの物体しか表現できないため，どちらか一方しか検出できません（課題で例を確認します）．

In [ ]:
S, B, C = 7, 2, n_class


def encode_yolo_target(boxes, labels, S=S, C=C, img_size=IMG_SIZE):
    """1枚の画像の (boxes, labels) を，S×Sグリッド上のターゲットに変換する"""
    cell_size = img_size / S
    obj_mask = torch.zeros(S, S, dtype=torch.bool)
    box_target = torch.zeros(S, S, 4)       # (x_cell, y_cell, w_img, h_img)
    box_target_xyxy = torch.zeros(S, S, 4)  # IoU計算用に，元のピクセル座標(xyxy)も保持しておく
    class_target = torch.zeros(S, S, C)

    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box.tolist()
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        w, h = x2 - x1, y2 - y1
        col = min(int(cx / cell_size), S - 1)
        row = min(int(cy / cell_size), S - 1)

        obj_mask[row, col] = True  # 1つのセルに複数の中心が入る場合は，後の物体で上書きされる
        box_target[row, col] = torch.tensor([
            cx / cell_size - col, cy / cell_size - row, w / img_size, h / img_size,
        ])
        box_target_xyxy[row, col] = torch.tensor([x1, y1, x2, y2])
        class_target[row, col] = 0.0
        class_target[row, col, label] = 1.0

    return obj_mask, box_target, box_target_xyxy, class_target


def build_batch_targets(targets):
    """バッチ内の各画像のtargetをencode_yolo_targetで変換し，バッチ次元でまとめる"""
    obj_masks, box_targets, box_targets_xyxy, class_targets = [], [], [], []
    for t in targets:
        om, bt, btx, ct = encode_yolo_target(t['boxes'], t['labels'])
        obj_masks.append(om)
        box_targets.append(bt)
        box_targets_xyxy.append(btx)
        class_targets.append(ct)

    return (torch.stack(obj_masks).to(device), torch.stack(box_targets).to(device),
            torch.stack(box_targets_xyxy).to(device), torch.stack(class_targets).to(device))


# 動作確認：学習データ1枚を実際に変換してみる
sample_image, sample_target = train_data[0]
om, bt, btx, ct = encode_yolo_target(sample_target['boxes'], sample_target['labels'])
print('物体を含むセル数:', om.sum().item(), '/ 画像内の物体数:', len(sample_target['labels']))

### 「Responsible」ボックスの選択
YOLOv1は各セルにつきB個のボックスを予測しますが，学習時に座標を回帰させるのは，そのうち**GTボックスとのIoUが最も高い1つだけ**です（これを「responsible」なボックスと呼びます）．どちらが responsible になるかは固定ではなく，学習中の予測値に応じて毎回変わる点に注意してください．

responsibleでない方のボックスは，座標の学習対象にはせず，confidence（信頼度）のみ「物体なし（0）」として学習します．IoUの計算には，予測ボックス・GTボックスをどちらも画像上のピクセル座標(xyxy)に変換したうえで`torchvision.ops.box_iou`を用います．

In [ ]:
def decode_pred_boxes(xy, wh, S=S, img_size=IMG_SIZE):
    """セル内相対(x, y)・画像相対(w, h)の予測を，ピクセル座標(xyxy)のボックスにデコードする
    xy, wh: (N, S, S, B, 2)"""
    cell_size = img_size / S
    row_idx = torch.arange(S, device=xy.device).float().view(1, S, 1, 1, 1)  # y方向（行）
    col_idx = torch.arange(S, device=xy.device).float().view(1, 1, S, 1, 1)  # x方向（列）

    cx = (col_idx + xy[..., 0:1]) * cell_size
    cy = (row_idx + xy[..., 1:2]) * cell_size
    w = wh[..., 0:1] * img_size
    h = wh[..., 1:2] * img_size

    boxes = torch.cat([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)
    return boxes  # (N, S, S, B, 4)


def compute_responsible_iou(pred_boxes_obj, gt_boxes_obj):
    """物体を含むM個のセルについて，B個の予測ボックスとGTボックスとのIoUを計算する
    pred_boxes_obj: (M, B, 4), gt_boxes_obj: (M, 4) -> (M, B)"""
    M, Bn = pred_boxes_obj.shape[:2]
    pred_flat = pred_boxes_obj.reshape(M * Bn, 4)
    gt_flat = gt_boxes_obj.unsqueeze(1).expand(M, Bn, 4).reshape(M * Bn, 4)
    # box_iouは全ペアのIoU行列を返すため，対応する行だけ（対角成分）を取り出す
    iou_matrix = box_iou(pred_flat, gt_flat)
    return torch.diagonal(iou_matrix).view(M, Bn)

## 損失関数（YOLOv1 Loss）
YOLOv1の損失は，SSDのようなSmooth L1やクロスエントロピーではなく，原論文に忠実に**すべて二乗和誤差（SSE）**で構成されます．物体を含むセルは全体のごく一部であるため，そのままでは「物体なし」の学習ばかりが支配的になってしまいます．そこで，座標の誤差に重み$\lambda_{coord}=5$を，物体を含まないセルのconfidence誤差に重み$\lambda_{noobj}=0.5$をかけてバランスを取ります．

損失は次の4項の和です．

1. **座標損失**：物体を含むセルのresponsibleなボックスのみ．x, yはそのまま，w, hは平方根を取ってから二乗誤差を計算します（大きい物体と小さい物体とで，誤差が結果に与える影響度を均等にするため）．
2. **confidence損失（物体あり）**：物体を含むセルのresponsibleなボックスについて，予測confidenceと，そのボックスの現在のIoU（固定の正解ラベルではなく，学習が進むにつれて変化する値）との二乗誤差．
3. **confidence損失（物体なし）**：物体を含まないセルの全ボックス，および物体を含むセルの非responsibleなボックスについて，予測confidenceと0との二乗誤差（$\lambda_{noobj}$で重み付け）．
4. **クラス損失**：物体を含むセルのみ，クラス確率と正解one-hotベクトルとの二乗誤差．クラス確率はB個のボックスで共有されているため，ボックスごとではなくセルごとに1回だけ計算します．

In [ ]:
class YOLOv1Loss(nn.Module):
    def __init__(self, S=7, B=2, C=20, lambda_coord=5.0, lambda_noobj=0.5):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj

    def forward(self, preds, targets):
        obj_mask, box_target, box_target_xyxy, class_target = targets
        N = preds.size(0)

        box_preds = preds[..., :self.B * 5].view(N, self.S, self.S, self.B, 5)
        class_preds = preds[..., self.B * 5:]  # (N, S, S, C)
        xy, wh, conf = box_preds[..., 0:2], box_preds[..., 2:4], box_preds[..., 4]

        pred_boxes_xyxy = decode_pred_boxes(xy, wh, S=self.S)  # (N, S, S, B, 4)

        obj_idx = obj_mask.nonzero(as_tuple=False)  # (M, 3) : n, row, col
        resp_mask = torch.zeros(N, self.S, self.S, self.B, dtype=torch.bool, device=preds.device)

        if obj_idx.numel() > 0:
            n_i, row_i, col_i = obj_idx[:, 0], obj_idx[:, 1], obj_idx[:, 2]
            m_range = torch.arange(obj_idx.size(0), device=preds.device)

            # 「responsible」なボックス = 現在の予測とGTのIoUが最大のボックス
            ious = compute_responsible_iou(pred_boxes_xyxy[n_i, row_i, col_i], box_target_xyxy[n_i, row_i, col_i])
            responsible = ious.argmax(dim=1)
            best_iou = ious[m_range, responsible]
            resp_mask[n_i, row_i, col_i, responsible] = True

            pred_xy = xy[n_i, row_i, col_i][m_range, responsible]
            pred_wh = wh[n_i, row_i, col_i][m_range, responsible].clamp(min=1e-6)
            gt_xy = box_target[n_i, row_i, col_i][:, 0:2]
            gt_wh = box_target[n_i, row_i, col_i][:, 2:4].clamp(min=1e-6)

            coord_loss = self.lambda_coord * (
                F.mse_loss(pred_xy, gt_xy, reduction='sum')
                + F.mse_loss(torch.sqrt(pred_wh), torch.sqrt(gt_wh), reduction='sum')
            )
            pred_conf_obj = conf[n_i, row_i, col_i][m_range, responsible]
            conf_obj_loss = F.mse_loss(pred_conf_obj, best_iou.detach(), reduction='sum')
            class_loss = F.mse_loss(class_preds[n_i, row_i, col_i], class_target[n_i, row_i, col_i], reduction='sum')
        else:
            coord_loss = conf_obj_loss = class_loss = torch.zeros((), device=preds.device)

        # 物体を含まないセルの全ボックス + 物体を含むセルの非responsibleなボックス
        obj_mask_b = obj_mask.unsqueeze(-1).expand(-1, -1, -1, self.B)
        noobj_mask = (~obj_mask_b) | (obj_mask_b & ~resp_mask)
        conf_noobj_loss = self.lambda_noobj * F.mse_loss(
            conf[noobj_mask], torch.zeros_like(conf[noobj_mask]), reduction='sum')

        total = (coord_loss + conf_obj_loss + conf_noobj_loss + class_loss) / N
        return total, coord_loss / N, conf_obj_loss / N, conf_noobj_loss / N, class_loss / N

## ネットワークの作成
`YOLOv1`をインスタンス化し，`.to(device)`でGPUに配置します．最適化手法は原論文と同じくモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）とします．事前学習済みバックボーンを使用しているため，スクラッチ学習のCIFAR100分類モデルよりも小さめの学習率です．

In [ ]:
model = YOLOv1(S=S, B=B, C=C).to(device)
criterion = YOLOv1Loss(S=S, B=B, C=C, lambda_coord=5.0, lambda_noobj=0.5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
ミニバッチサイズ8，学習エポック数20とします．バッチ内の各画像のGTを`build_batch_targets`でグリッドターゲットに変換したのち，ネットワークの出力と合わせて4項のSSE損失を計算します．学習の進み方を確認しやすいよう，各損失項の値も合わせて表示します．

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = sum_coord = sum_conf_obj = sum_conf_noobj = sum_class = 0.0

    for images, targets in train_loader:
        images = images.to(device)
        batch_targets = build_batch_targets(targets)

        preds = model(images)
        loss, coord_loss, conf_obj_loss, conf_noobj_loss, class_loss = criterion(preds, batch_targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        sum_coord += coord_loss.item()
        sum_conf_obj += conf_obj_loss.item()
        sum_conf_noobj += conf_noobj_loss.item()
        sum_class += class_loss.item()

    n_batches = len(train_loader)
    print("epoch: {}, mean loss: {:.4f} (coord: {:.4f}, conf_obj: {:.4f}, conf_noobj: {:.4f}, class: {:.4f}), elapsed time: {:.1f}".format(
        epoch, sum_loss / n_batches, sum_coord / n_batches, sum_conf_obj / n_batches,
        sum_conf_noobj / n_batches, sum_class / n_batches, time() - train_start))

## 推論（デコード・NMS）
学習済みモデルから最終的な検出結果を得るための関数を定義します．各セル・各ボックスについて，`box_confidence × class_probability`をクラスごとのスコアとして算出し（`class_probability`はセル内でB個のボックスに共有されている点に注意），ピクセル座標にデコードしたのち，閾値処理とクラスごとの`torchvision.ops.nms`で重複した検出結果を除去します．

In [ ]:
def predict(model, image_tensor, conf_threshold=0.1, nms_threshold=0.45):
    model.eval()
    with torch.no_grad():
        preds = model(image_tensor.unsqueeze(0).to(device))  # (1, S, S, B*5+C)

    box_preds = preds[0, ..., :B * 5].view(S, S, B, 5)
    class_preds = preds[0, ..., B * 5:]  # (S, S, C)：セルごとに共有された1組のクラス確率

    xy = box_preds[..., 0:2].unsqueeze(0)
    wh = box_preds[..., 2:4].unsqueeze(0)
    conf = box_preds[..., 4]  # (S, S, B)

    boxes_xyxy = decode_pred_boxes(xy, wh)[0].clamp(min=0, max=IMG_SIZE)  # (S, S, B, 4)

    # クラスごとのスコア = ボックスconfidence × セルのクラス確率 -> (S, S, B, C)
    class_scores = conf.unsqueeze(-1) * class_preds.unsqueeze(2)

    boxes_flat = boxes_xyxy.reshape(-1, 4)
    scores_flat = class_scores.reshape(-1, C)

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(C):
        cls_scores = scores_flat[:, class_idx]
        mask = cls_scores > conf_threshold
        if mask.sum() == 0:
            continue
        cls_boxes = boxes_flat[mask]
        cls_scores = cls_scores[mask]

        keep = nms(cls_boxes, cls_scores, nms_threshold)
        result_boxes.append(cls_boxes[keep])
        result_scores.append(cls_scores[keep])
        result_labels.append(torch.full((len(keep),), class_idx, dtype=torch.long))

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
SSDと同様，`torchmetrics`の`MeanAveragePrecision`を用いてmAPを算出します（自前でPrecision-Recall曲線からAPを計算しない）．VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認します（NMS・IoU計算の詳細は`ssd.ipynb`を参照）．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画します．

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルYOLOv1との違い

| | 原論文 | 本ノートブック |
|---|---|---|
| バックボーン | 独自の24層Darknet系CNN（ImageNetで224×224の分類タスクを事前学習） | ResNet50（ImageNet事前学習済み，`layer4`まで使用） |
| 検出ヘッド | 2層の全結合層（4096次元を経由） | 1×1畳み込みによる全畳み込みヘッド |
| 出力の活性化関数 | 線形出力（活性化なし）にSSEを直接適用 | 座標・confidenceにSigmoid，クラス確率にSoftmaxを適用してから学習を安定化 |
| 学習の解像度 | 224×224で分類を事前学習した後，448×448にファインチューン（2段階） | 事前学習済みResNet50を448×448に直接適用（1段階） |
| 中間層の活性化関数 | Leaky ReLU（全層） | ResNet部分はReLU，追加した層のみLeaky ReLU |
| NMS・IoU計算 | 独自実装 | `torchvision.ops`の`nms`・`box_iou`を使用 |

グリッドサイズS=7，ボックス数B=2，クラス確率をセルごとに共有する設計，4項のSSE損失（$\lambda_{coord}=5$, $\lambda_{noobj}=0.5$），responsibleなボックスの選択など，YOLOv1の核となる部分は原論文に忠実に実装しています．

## 課題

1. `B`（1セルあたりの予測ボックス数）を1や3に変更し，出力の形状・パラメータ数・検出精度への影響を確認してみましょう．

2. `YOLOv1Loss`の`lambda_coord`・`lambda_noobj`を変更し，学習の安定性（各損失項の値の推移）にどのような影響が出るか確認してみましょう．

3. 小さな物体が密集している画像（`test_data`の中から探す，またはbboxが近接する例を自作する）を用いて，2つの物体の中心が同じグリッドセルに入ってしまい，どちらか一方しか検出できないケースを実際に確認してみましょう．